# Notebook 05 — Transformer Encoder Block

## Learning Objectives
- Understand how Multi-Head Attention and Feed-Forward layers combine into one encoder block.
- Use `TransformerEncoderBlock` and `TransformerEncoder` from `src.models.transformer_encoder`.
- Build a minimal sequence classification task to train the encoder end-to-end.
- Plot training loss and accuracy curves.
- Verify all tensor shapes at each step of the forward pass.

## Big Picture
The Transformer encoder block is the reusable unit that powers BERT, ViT, and many other models.
Each block consists of:
1. Layer normalization
2. Multi-head self-attention + residual
3. Layer normalization
4. Position-wise feed-forward network + residual

Stacking N of these blocks builds a deep Transformer encoder.

## 1. Imports and Setup

In [1]:
import sys
sys.path.insert(0, '..')

import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.utils.seed import seed_everything
from src.utils.device import get_best_device
from src.utils.cleanup import prepare_run_dir
from src.utils.reporting import save_table_csv, save_metrics_json
from src.utils.paths import RUNS_ATTENTION_DIR
from src.models.transformer_encoder import TransformerEncoderBlock, TransformerEncoder
from src.models.positional_encoding import SinusoidalPositionalEncoding

seed_everything(42)
device = get_best_device()
prepare_run_dir(RUNS_ATTENTION_DIR, clean=False)
print(f"Device: {device}")
print("All imports successful.")

[seed] Random seed set to 42.
Device: cuda
All imports successful.


## 2. Dataset: Synthetic First-Last-Token Matching Task

We create a synthetic sequence classification task:
- Sequences of length 12 with tokens from a vocabulary of 50.
- **Label = 1** if the first token equals the last token, **Label = 0** otherwise.

This is a good test for the encoder because it requires the model to compare
distant positions — exactly what attention is good at.

In [2]:
# Hyperparameters
VOCAB_SIZE = 50
SEQ_LEN    = 12
D_MODEL    = 64
NUM_HEADS  = 4
NUM_LAYERS = 2
DIM_FF     = 128
BATCH_SIZE = 32
N_TRAIN    = 1000
N_VAL      = 200
NUM_EPOCHS = 25
LR         = 1e-3

torch.manual_seed(42)

def make_dataset(n_samples, vocab_size, seq_len, seed=0):
    """Create sequences and labels: label=1 if seq[0] == seq[-1]."""
    rng = torch.Generator().manual_seed(seed)

    # Token IDs must stay in [1, vocab_size - 1].
    # ID 0 is reserved for padding.
    seqs = torch.randint(1, vocab_size, (n_samples, seq_len), generator=rng)

    half = n_samples // 2

    # Positive samples: first token equals last token.
    seqs[:half, -1] = seqs[:half, 0]

    # Negative samples: last token is guaranteed different from first token,
    # while still staying inside the valid embedding range [1, vocab_size - 1].
    seqs[half:, -1] = (seqs[half:, 0] % (vocab_size - 1)) + 1

    labels = (seqs[:, 0] == seqs[:, -1]).long()
    return seqs, labels


train_seqs, train_labels = make_dataset(N_TRAIN, VOCAB_SIZE, SEQ_LEN, seed=42)
val_seqs,   val_labels   = make_dataset(N_VAL,   VOCAB_SIZE, SEQ_LEN, seed=99)

# Safety checks: embedding accepts token IDs from 0 to VOCAB_SIZE - 1.
assert train_seqs.min().item() >= 1
assert val_seqs.min().item() >= 1
assert train_seqs.max().item() < VOCAB_SIZE
assert val_seqs.max().item() < VOCAB_SIZE

train_loader = DataLoader(
    TensorDataset(train_seqs, train_labels),
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    TensorDataset(val_seqs, val_labels),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print(f"Train samples : {N_TRAIN}  |  Val samples: {N_VAL}")
print(f"Sequence shape: {train_seqs.shape}")
print(f"Label distribution: {train_labels.float().mean().item():.2f} positive")
print(f"Token ID range : train=[{train_seqs.min().item()}, {train_seqs.max().item()}], "
      f"val=[{val_seqs.min().item()}, {val_seqs.max().item()}]")
print(f"Example sequence: {train_seqs[0].tolist()}")
print(f"Example label   : {train_labels[0].item()} "
      f"(first={train_seqs[0,0]}, last={train_seqs[0,-1]})")

Train samples : 1000  |  Val samples: 200
Sequence shape: torch.Size([1000, 12])
Label distribution: 0.50 positive
Token ID range : train=[1, 49], val=[1, 49]
Example sequence: [30, 17, 12, 13, 47, 18, 20, 20, 8, 22, 22, 30]
Example label   : 1 (first=30, last=30)


## 3. Architecture (Text Diagram)

```
Input IDs [batch, seq_len]
        │
  Embedding [batch, seq_len, d_model]   (vocab_size=50, d_model=64)
        │
  Sinusoidal PE  (adds position info)
        │
  ┌─────────────────────────────────────┐
  │         Encoder Block ×2            │
  │  ┌──────────────────────────────┐   │
  │  │ LayerNorm                    │   │
  │  │ Multi-Head Self-Attention    │   │
  │  │ + Residual                   │   │
  │  │ LayerNorm                    │   │
  │  │ FeedForward (d_model→256→64) │   │
  │  │ + Residual                   │   │
  │  └──────────────────────────────┘   │
  └─────────────────────────────────────┘
        │
  Mean Pool → [batch, d_model]
        │
  Linear → [batch, 2]   (binary classifier)
```

## 4. Theory: Pre-Norm vs Post-Norm

The original Transformer (Vaswani et al.) uses **post-norm** (LayerNorm after the residual).
Modern practice prefers **pre-norm** (LayerNorm before the sub-layer):

- **Pre-norm**: `x = x + Attn(LayerNorm(x))` — more stable gradients, trains without warmup.
- **Post-norm**: `x = LayerNorm(x + Attn(x))` — potentially higher quality ceiling, but tricky to train.

Our `TransformerEncoderBlock` uses pre-norm for robustness.

**Residual connections** ensure gradients flow directly back to early layers, enabling very deep stacks.

## 5. Implementation from Scratch

In [3]:
# Inspect the encoder block
block = TransformerEncoderBlock(
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    dim_feedforward=DIM_FF,
    dropout=0.1,
).to(device)
print("TransformerEncoderBlock:")
print(block)
n_params_block = sum(p.numel() for p in block.parameters())
print(f"\nParameters per block: {n_params_block:,}")

# Single forward pass through one block
dummy = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL).to(device)
block.eval()
with torch.no_grad():
    out_block, attn_w = block(dummy, return_weights=True)

print(f"\nInput  shape : {dummy.shape}    (batch, seq_len, d_model)")
print(f"Output shape : {out_block.shape}  (batch, seq_len, d_model)")
print(f"Attn weights : {attn_w.shape}   (batch, heads, seq_q, seq_k)")

TransformerEncoderBlock:
TransformerEncoderBlock(
  (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (self_attention): MultiHeadAttention(
    (w_q): Linear(in_features=64, out_features=64, bias=False)
    (w_k): Linear(in_features=64, out_features=64, bias=False)
    (w_v): Linear(in_features=64, out_features=64, bias=False)
    (w_o): Linear(in_features=64, out_features=64, bias=False)
    (attention): ScaledDotProductAttention(
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (dropout1): Dropout(p=0.1, inplace=False)
  (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (feed_forward): FeedForward(
    (linear1): Linear(in_features=64, out_features=128, bias=True)
    (activation): GELU(approximate='none')
    (dropout1): Dropout(p=0.1, inplace=False)
    (linear2): Linear(in_features=128, out_features=64, bias=True)
    (dropout2): Dropout(p=0.1, inplace=False)
  )
  (dropout2): Dropout(p=0.1, inplace=False)
)

Parameters per block: 33,216

Inpu

In [4]:
# Build the full classification model
class SequenceClassifier(nn.Module):
    """Encoder-based binary classifier: label=1 if first token == last token."""

    def __init__(self, vocab_size, num_classes, d_model, num_heads, num_layers,
                 dim_feedforward, seq_len, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_enc = SinusoidalPositionalEncoding(d_model, seq_len + 1, dropout)
        self.encoder = TransformerEncoder(
            d_model=d_model,
            num_heads=num_heads,
            num_layers=num_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
        )

        # first token + last token + absolute difference
        self.classifier = nn.Linear(d_model * 3, num_classes)

    def forward(self, input_ids):
        x = self.embedding(input_ids)      # [batch, seq_len, d_model]
        x = self.pos_enc(x)                # [batch, seq_len, d_model]
        x, _ = self.encoder(x)             # [batch, seq_len, d_model]

        first = x[:, 0, :]                 # [batch, d_model]
        last = x[:, -1, :]                 # [batch, d_model]
        features = torch.cat(
            [first, last, torch.abs(first - last)],
            dim=-1,
        )                                  # [batch, d_model * 3]

        return self.classifier(features)   # [batch, num_classes]


model = SequenceClassifier(
    vocab_size=VOCAB_SIZE,
    num_classes=2,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FF,
    seq_len=SEQ_LEN,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,}")

# Quick forward pass
sample_ids = train_seqs[:4].to(device)
with torch.no_grad():
    sample_out = model(sample_ids)
print(f"Input shape   : {sample_ids.shape}")
print(f"Output logits : {sample_out.shape}  (batch, num_classes)")

Model parameters: 70,146
Input shape   : torch.Size([4, 12])
Output logits : torch.Size([4, 2])  (batch, num_classes)


## 6. Sanity Checks

In [5]:
# Tensor shape checks
assert out_block.shape == (BATCH_SIZE, SEQ_LEN, D_MODEL), f"{out_block.shape}"
assert attn_w.shape == (BATCH_SIZE, NUM_HEADS, SEQ_LEN, SEQ_LEN), f"{attn_w.shape}"
assert sample_out.shape == (4, 2), f"{sample_out.shape}"
print("Shape checks passed.")

# Residual connection check: output should not be zero-mean
out_mean = out_block.mean().item()
out_std  = out_block.std().item()
print(f"Encoder block output — mean: {out_mean:.4f}, std: {out_std:.4f}")
assert out_std > 0.01, "Output has no variance — something is wrong!"

# Random init logits should be near zero
assert sample_out.abs().max().item() < 10, "Logits seem unreasonably large at init"
print("Sanity checks passed!")

Shape checks passed.
Encoder block output — mean: 0.0073, std: 1.0331
Sanity checks passed!


## 7. Training

In [6]:
# Training loop
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Training ──
    model.train()
    epoch_loss, correct, total = 0.0, 0, 0
    for seqs, labels in train_loader:
        seqs, labels = seqs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(seqs)                 # [batch, 2]
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
        correct += (logits.argmax(dim=-1) == labels).sum().item()
        total   += labels.size(0)
    train_losses.append(epoch_loss / len(train_loader))
    train_accs.append(correct / total)

    # ── Validation ──
    model.eval()
    val_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for seqs, labels in val_loader:
            seqs, labels = seqs.to(device), labels.to(device)
            logits = model(seqs)
            val_loss += criterion(logits, labels).item()
            v_correct += (logits.argmax(dim=-1) == labels).sum().item()
            v_total   += labels.size(0)
    val_losses.append(val_loss / len(val_loader))
    val_accs.append(v_correct / v_total)

    print(f"Epoch {epoch}/{NUM_EPOCHS}  "
          f"train_loss={train_losses[-1]:.4f}  train_acc={train_accs[-1]:.4f}  "
          f"val_loss={val_losses[-1]:.4f}  val_acc={val_accs[-1]:.4f}")

print("\nTraining complete!")

Epoch 1/25  train_loss=0.4781  train_acc=0.7990  val_loss=0.1865  val_acc=0.9950
Epoch 2/25  train_loss=0.1699  train_acc=0.9920  val_loss=0.0688  val_acc=1.0000
Epoch 3/25  train_loss=0.0748  train_acc=0.9990  val_loss=0.0199  val_acc=1.0000
Epoch 4/25  train_loss=0.0447  train_acc=0.9990  val_loss=0.0150  val_acc=1.0000
Epoch 5/25  train_loss=0.0279  train_acc=0.9990  val_loss=0.0071  val_acc=1.0000
Epoch 6/25  train_loss=0.0186  train_acc=1.0000  val_loss=0.0036  val_acc=1.0000
Epoch 7/25  train_loss=0.0144  train_acc=1.0000  val_loss=0.0035  val_acc=1.0000
Epoch 8/25  train_loss=0.0115  train_acc=1.0000  val_loss=0.0020  val_acc=1.0000
Epoch 9/25  train_loss=0.0086  train_acc=1.0000  val_loss=0.0015  val_acc=1.0000
Epoch 10/25  train_loss=0.0065  train_acc=1.0000  val_loss=0.0010  val_acc=1.0000
Epoch 11/25  train_loss=0.0052  train_acc=1.0000  val_loss=0.0008  val_acc=1.0000
Epoch 12/25  train_loss=0.0046  train_acc=1.0000  val_loss=0.0009  val_acc=1.0000
Epoch 13/25  train_loss=0

## 8. Evaluation

In [7]:
# Final evaluation on validation set
model.eval()
all_preds, all_labels_list = [], []
with torch.no_grad():
    for seqs, labels in val_loader:
        seqs = seqs.to(device)
        logits = model(seqs)
        all_preds.extend(logits.argmax(dim=-1).cpu().tolist())
        all_labels_list.extend(labels.tolist())

final_acc = sum(p == l for p, l in zip(all_preds, all_labels_list)) / len(all_labels_list)
print(f"Final validation accuracy: {final_acc:.4f}  ({final_acc*100:.1f}%)")
print(f"Random baseline          : 0.5000  (50.0%)")

# Save metrics
metrics = {
    'final_val_accuracy': final_acc,
    'final_val_loss': val_losses[-1],
    'final_train_loss': train_losses[-1],
    'num_epochs': NUM_EPOCHS,
    'num_params': n_params,
    'd_model': D_MODEL,
    'num_heads': NUM_HEADS,
    'num_layers': NUM_LAYERS,
}
metrics_path = RUNS_ATTENTION_DIR / 'encoder_metrics.json'
save_metrics_json(metrics, metrics_path)
print(f"Metrics saved to: {metrics_path}")

Final validation accuracy: 1.0000  (100.0%)
Random baseline          : 0.5000  (50.0%)
Metrics saved to: /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/attention/encoder_metrics.json


## 9. Visualization

In [8]:
# Plot training curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

epochs_range = range(1, NUM_EPOCHS + 1)

ax1.plot(epochs_range, train_losses, 'o-', label='Train', color='steelblue')
ax1.plot(epochs_range, val_losses,   's-', label='Val',   color='tomato')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Training Loss Curve')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, [a*100 for a in train_accs], 'o-', label='Train', color='steelblue')
ax2.plot(epochs_range, [a*100 for a in val_accs],   's-', label='Val',   color='tomato')
ax2.axhline(50, color='gray', linestyle='--', alpha=0.5, label='Random baseline')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training Accuracy Curve')
ax2.set_ylim(0, 105)
ax2.legend()
ax2.grid(True, alpha=0.3)

fig.suptitle(
    f'TransformerEncoder: First-Last Token Match\n'
    f'd_model={D_MODEL}, heads={NUM_HEADS}, layers={NUM_LAYERS}',
    fontsize=12
)
fig.tight_layout()

curve_path = RUNS_ATTENTION_DIR / 'encoder_training_curve.png'
fig.savefig(curve_path, dpi=120)
plt.close(fig)
print(f"Training curve saved to: {curve_path}")

Training curve saved to: /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/attention/encoder_training_curve.png


In [9]:
# Save shape info to CSV
shape_rows = [
    {'step': '1. Token IDs',          'shape': f'(batch={BATCH_SIZE}, seq={SEQ_LEN})',              'note': 'Input to embedding'},
    {'step': '2. Token Embeddings',   'shape': f'({BATCH_SIZE}, {SEQ_LEN}, {D_MODEL})',             'note': 'After embedding lookup'},
    {'step': '3. + Positional Enc',   'shape': f'({BATCH_SIZE}, {SEQ_LEN}, {D_MODEL})',             'note': 'Same shape, different values'},
    {'step': '4. After Encoder×2',    'shape': f'({BATCH_SIZE}, {SEQ_LEN}, {D_MODEL})',             'note': 'Shape preserved through blocks'},
    {'step': '5. Mean Pool',          'shape': f'({BATCH_SIZE}, {D_MODEL})',                         'note': 'Average over seq_len dim'},
    {'step': '6. Classifier logits',  'shape': f'({BATCH_SIZE}, 2)',                                 'note': 'Binary classification output'},
]
csv_path = RUNS_ATTENTION_DIR / 'encoder_shapes.csv'
save_table_csv(shape_rows, csv_path)
print(f"Shape info saved to: {csv_path}")
for row in shape_rows:
    print(f"  {row['step']:<28} {row['shape']:<35} {row['note']}")

Shape info saved to: /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/attention/encoder_shapes.csv
  1. Token IDs                 (batch=32, seq=12)                  Input to embedding
  2. Token Embeddings          (32, 12, 64)                        After embedding lookup
  3. + Positional Enc          (32, 12, 64)                        Same shape, different values
  4. After Encoder×2           (32, 12, 64)                        Shape preserved through blocks
  5. Mean Pool                 (32, 64)                            Average over seq_len dim
  6. Classifier logits         (32, 2)                             Binary classification output


## 10. Failure Cases

**Model doesn't learn (accuracy stays ~50%)**: Check the task is not too hard for the model size. With d_model=8 and num_layers=1, the model may be too small. Increase d_model to 64.

**Exploding loss**: Missing gradient clipping. Always use `torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)` in early experiments.

**Padding ignored**: If sequences have padding tokens, pass `src_key_padding_mask` to the encoder. Without it, the model attends to padding positions — irrelevant noise.

**Pre-norm vs post-norm**: Swapping norm order (pre→post) with high learning rates causes instability. Pre-norm is safer for quick experiments.

## 11. Exercises

1. **Depth ablation**: Train with num_layers=1 and num_layers=4. Plot all three training curves. At what depth does accuracy peak?
2. **Mean pooling vs first token**: Replace mean pooling with just taking the first token output `x[:, 0, :]`. Does it still work? Why or why not?
3. **Harder task**: Modify `make_dataset` so label=1 if the *second* token equals the *second-to-last* token. Does the encoder still solve it?
4. **Attention visualization**: After training, extract attention weights for a positive example (first == last). Do the first/last positions attend to each other?
5. **Return weights from encoder**: Use `TransformerEncoder(return_all_weights=True)`. Plot the attention from the last layer's first head.

## 12. Key Takeaways

- **TransformerEncoderBlock** = LayerNorm + MultiHeadSelfAttention + residual + LayerNorm + FFN + residual.
- **Pre-norm style** (norm before sub-layer) is more training-stable than post-norm.
- **Shape contract**: encoder preserves the sequence shape `[batch, seq_len, d_model]` all the way through.
- **Mean pooling** over the sequence dimension gives a fixed-size sentence representation.
- A 2-layer encoder with d_model=64 can easily learn to compare distant tokens in a short sequence.